In [ ]:
import os, sys
project_root = os.path.abspath('..').replace('\\', '/')
if project_root not in [p.replace('\\', '/') for p in sys.path]:
    sys.path.append(project_root)


# 09 规则引擎 (core.rules)

基于 `hscredit.xlsx` 真实信贷数据，演示规则创建/评估/挖掘/优化。

In [5]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import hscredit as hsc
from hscredit import (init_setting, Rule,
    get_columns_from_query, optimize_expr, beautify_expr, get_expr_variables,
    ruleset_report,
)
from hscredit.core.rules.mining import SingleFeatureRuleMiner, MultiFeatureRuleMiner
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D')
target = 'FPD15'
features = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','overdue_loan_m1_count_12m',
    'overdue_loan_m0_count_12m','loan_count_12m','loan_count_3m','loan_count_1m',
    'network_loan_lender_count','lender_inquiry_count_3m','hit_lender_count']
df_clean = df[features + [target]].fillna(-999)
print(f'样本数: {len(df):,}, 坏率: {df[target].mean():.2%}')

ModuleNotFoundError: No module named 'hscredit'

## 1. Rule 对象 — 信贷业务规则

In [3]:
# 典型信贷风控规则
rules_business = [
    ('低评分拒绝',   'bj_qy24 < 550'),
    ('多头借贷',     'lender_count_12m > 20'),
    ('近期逾期',     'overdue_loan_m1_count_12m >= 2'),
    ('高频申请',     'lender_inquiry_count_3m > 15'),
    ('组合规则',     'bj_qy24 < 580 and overdue_loan_m0_count_12m > 0'),
]
result_rows = []
for name, expr in rules_business:
    try:
        r = Rule(expr)
        hit = r.apply(df_clean)
        result_rows.append({
            '规则名称': name, '规则表达式': expr,
            '命中率': round(hit.mean(),4),
            '命中坏率': round(df_clean.loc[hit, target].mean(),4),
            '未命中坏率': round(df_clean.loc[~hit, target].mean(),4),
            'LIFT': round(df_clean.loc[hit, target].mean() / df_clean[target].mean(), 2) if df_clean[target].mean() > 0 else 0
        })
    except Exception as e:
        print(f'{name}: {e}')
pd.DataFrame(result_rows)

低评分拒绝: name 'Rule' is not defined
多头借贷: name 'Rule' is not defined
近期逾期: name 'Rule' is not defined
高频申请: name 'Rule' is not defined
组合规则: name 'Rule' is not defined


""


## 2. ruleset_report — 规则集综合评估

In [ ]:
rule_exprs = [expr for _, expr in rules_business]
report = ruleset_report(df_clean, rules=rule_exprs, target=target)
print(report.columns.tolist())
report

## 3. 表达式工具

In [ ]:
expr = '(bj_qy24 < 580) AND (overdue_loan_m1_count_12m >= 1 OR lender_count_12m > 15)'
print('原始表达式:', expr)
print('优化后:    ', optimize_expr(expr))
print('美化后:    ', beautify_expr(expr))
print('涉及变量:  ', get_expr_variables(expr))
print('涉及列名:  ', get_columns_from_query(expr, df_clean))

## 4. 单特征规则挖掘

In [ ]:
miner = SingleFeatureRuleMiner(
    target=target,
    min_lift=2.0,
    min_support=0.03,
    max_n_bins=8,
)
miner.fit(df_clean, features=features)
results = miner.rules_df_
print(f'挖掘到 {len(results)} 条单特征规则')
results.sort_values('lift', ascending=False).head(15)

## 5. 多特征组合规则挖掘

In [ ]:
multi_miner = MultiFeatureRuleMiner(
    target=target,
    min_lift=3.0,
    min_support=0.02,
    max_depth=2,
    max_n_bins=5,
)
multi_miner.fit(df_clean, features=features[:6])
print(f'多特征规则: {len(multi_miner.rules_df_)} 条')
multi_miner.rules_df_.sort_values('lift', ascending=False).head(10)

## 6. 规则命中分析 — 叠加效果

In [ ]:
# 计算多条规则OR/AND组合的效果
hit_any = pd.Series(False, index=df_clean.index)
hit_all = pd.Series(True, index=df_clean.index)
for expr in rule_exprs[:3]:
    h = Rule(expr).apply(df_clean)
    hit_any |= h
    hit_all &= h
print(f'任一命中: 命中率={hit_any.mean():.2%}, 坏率={df_clean.loc[hit_any,target].mean():.2%}')
print(f'全部命中: 命中率={hit_all.mean():.2%}, 坏率={df_clean.loc[hit_all,target].mean():.2%}')